# Business Idea Evaluator - Colab Demo

A **human-in-the-loop, parallelized multi-agent system** that evaluates a startup idea from four professional angles (Market, Legal, Technical, Strategy) using **LangGraph** + **Gemini**.

**How to run:** click *Runtime -> Run all*, paste your free [Google AI Studio API key](https://aistudio.google.com/apikey) when asked, then type your business idea.

Repo: https://github.com/ranarefaat365-code/business-idea-evaluator

## 1. Install dependencies

In [ ]:
!pip install -q langgraph langchain-google-genai langchain-core

## 2. Enter your API key

Your key is entered privately with `getpass` - it is **not** saved in the notebook.

In [ ]:
import os
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Paste your Google AI Studio API key: ")
MODEL_NAME = "gemini-3.6-flash"

## 3. Define the LLM, state, nodes, and graph

In [ ]:
import operator
from typing import List, Annotated, Dict
from typing_extensions import TypedDict

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, BaseMessage
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, END
from IPython.display import Image, display, Markdown

llm = ChatGoogleGenerativeAI(model=MODEL_NAME)


class State(TypedDict):
    idea: str
    messages: Annotated[List[BaseMessage], add_messages]
    advisor_reports: Annotated[Dict[str, str], operator.or_]
    final_report: str


def as_text(content) -> str:
    """Gemini may return content as a list of parts; normalize to a string."""
    if isinstance(content, list):
        out = []
        for p in content:
            if isinstance(p, str): out.append(p)
            elif isinstance(p, dict): out.append(p.get("text", ""))
        return "".join(out)
    return content or ""

### Human-in-the-loop nodes

In [ ]:
base_system_msg = SystemMessage(content="""
You are a helpful assistant.
Your job: decide whether you have enough information about the start-up idea.
If not, ask ONE precise follow-up question.
If yes, respond with exactly: DONE
""")

def decide_node(state: State):
    conversation = [base_system_msg] + state["messages"]
    return {"messages": [llm.invoke(conversation)]}

def route(state: State):
    last = state["messages"][-1]
    return "fanout" if as_text(last.content).strip().upper().startswith("DONE") else "ask_user_node"

def ask_user_node(state: State):
    question = as_text(state["messages"][-1].content)
    print(f"\nAssistant: {question}\n")
    human = input("You: ")
    return {"messages": [HumanMessage(content=human)]}

### Advisor nodes (run in parallel)

In [ ]:
ADVISOR_PROMPTS = {
    "Market Analyst": "You are a senior MARKET ANALYST. Evaluate market potential, competition, "
        "target demographics, and trends: market sizing, competitor research, customer segments, timing/macro.",
    "Legal Advisor": "You are a LEGAL ADVISOR. Identify IP/licensing/trademark needs, spot compliance "
        "issues (e.g. GDPR), and evaluate contract/partnership considerations.",
    "Technical Advisor": "You are a TECHNICAL/PRODUCT FEASIBILITY EXPERT. Estimate development complexity/time, "
        "recommend tech stacks, and evaluate infrastructure/scalability/cost risk.",
    "Strategist Advisor": "You are a STRATEGIST ADVISOR. Define launch milestones, select distribution channels "
        "and positioning, and craft early traction tactics.",
}

def run_advisor(role, state):
    prompt = f"{ADVISOR_PROMPTS[role]}\n\nIdea / conversation so far:\n{state['messages']}"
    report = llm.invoke([HumanMessage(content=prompt)])
    return {"advisor_reports": {role: report.content}}

def market_analyst_advisor(state): return run_advisor("Market Analyst", state)
def legal_advisor(state):          return run_advisor("Legal Advisor", state)
def technical_advisor(state):      return run_advisor("Technical Advisor", state)
def strategist_advisor(state):     return run_advisor("Strategist Advisor", state)

### Collect & report, then build the graph

In [ ]:
def collect_and_report(state: State):
    if len(state["advisor_reports"]) < 4:
        return {}
    prompt = ("You are a senior consultant. Combine the advisor notes below into one "
              f"clear, structured evaluation report for the founder.\n\n{state['advisor_reports']}")
    report = as_text(llm.invoke([HumanMessage(content=prompt)]).content)
    display(Markdown("## FINAL REPORT\n\n" + report))
    return {"final_report": report}


builder = StateGraph(State)
builder.add_node("decide_node", decide_node)
builder.add_node("ask_user_node", ask_user_node)
builder.add_node("fanout", lambda s: {})
builder.add_node("market_analyst_advisor", market_analyst_advisor)
builder.add_node("legal_advisor", legal_advisor)
builder.add_node("technical_advisor", technical_advisor)
builder.add_node("strategist_advisor", strategist_advisor)
builder.add_node("collect_and_report", collect_and_report)

builder.set_entry_point("decide_node")
builder.add_edge("ask_user_node", "decide_node")
builder.add_conditional_edges("decide_node", route,
    {"ask_user_node": "ask_user_node", "fanout": "fanout"})
for a in ["market_analyst_advisor","legal_advisor","technical_advisor","strategist_advisor"]:
    builder.add_edge("fanout", a)
    builder.add_edge(a, "collect_and_report")
builder.add_edge("collect_and_report", END)

graph = builder.compile()
display(Image(graph.get_graph(xray=1).draw_mermaid_png()))

## 4. Run it

Tip to try: *A subscription box that delivers healthy, portion-controlled Egyptian meals to busy professionals in Cairo and Alexandria.*

In [ ]:
print("What is your business idea?")
idea = input("You: ")
init_state: State = {"idea": idea, "messages": [HumanMessage(content=idea)],
                     "advisor_reports": {}, "final_report": ""}
graph.invoke(init_state, config={"configurable": {"thread_id": "run-1"}});